# 03 Raster Preprocessing

## Precipitation Downscaling — Khulna

Objectives:

- Select a reference grid
- Reproject raster datasets
- Resample raster datasets
- Clip rasters to the Khulna study area
- Align all rasters to the same grid
- Export processed rasters
- Verify raster consistency

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import yaml

from rasterio.mask import mask
from rasterio.warp import reproject, Resampling

In [2]:
PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna"
)

with open(
    PROJECT_ROOT / "environment.yml",
    "r",
    encoding="utf-8"
) as file:
    config = yaml.safe_load(file)

print("Configuration loaded successfully.")
print("Project root:", PROJECT_ROOT)

Configuration loaded successfully.
Project root: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna


In [3]:
RAW_DATA = PROJECT_ROOT / config["paths"]["raw_data"]
PROCESSED_DATA = PROJECT_ROOT / config["paths"]["processed_data"]

CCS = PROJECT_ROOT / config["paths"]["ccs"]
CDR = PROJECT_ROOT / config["paths"]["cdr"]
CHIRPS = PROJECT_ROOT / config["paths"]["chirps"]
ERA5 = PROJECT_ROOT / config["paths"]["era5"]

GSMAP = PROJECT_ROOT / config["paths"]["gsmap"]
GSMAP_MVK = PROJECT_ROOT / config["paths"]["gsmap_mvk"]
IMERG = PROJECT_ROOT / config["paths"]["imerg"]

LST = PROJECT_ROOT / config["paths"]["lst"]
NDVI = PROJECT_ROOT / config["paths"]["ndvi"]
PDIR = PROJECT_ROOT / config["paths"]["pdir"]
PERSIANN = PROJECT_ROOT / config["paths"]["persiann"]

LAND_VARIABLE = PROJECT_ROOT / config["paths"]["land_variable"]
DISTANCE_SEA = PROJECT_ROOT / config["paths"]["distance_sea"]
BOUNDARY = PROJECT_ROOT / config["paths"]["boundary"]

RASTER_OUTPUT = PROCESSED_DATA / "rasters"
RASTER_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Raster output:", RASTER_OUTPUT)

Raster output: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters


In [4]:
boundary_files = sorted(BOUNDARY.glob("*.shp"))

if not boundary_files:
    raise FileNotFoundError(
        f"No shapefile found in {BOUNDARY}"
    )

boundary_file = boundary_files[0]

khulna_boundary = gpd.read_file(boundary_file)

print("Boundary file:", boundary_file.name)
print("Boundary CRS:", khulna_boundary.crs)
print("Features:", len(khulna_boundary))

khulna_boundary.head()

Boundary file: Khulna.shp
Boundary CRS: EPSG:4326
Features: 1


,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_n,center_lat,center_lon,geometry
0,Khulna,None,None,None,BD4047,Khulna,None,None,None,BD40,...,4457.26501,v03,en,None,None,None,None,22.365739,89.452816,"POLYGON ((89.45595 23.01135, 89.45661 23.01113..."


In [5]:
khulna_boundary = khulna_boundary.dissolve()

print("Dissolved features:", len(khulna_boundary))

Dissolved features: 1


In [6]:
khulna_boundary

,geometry,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,...,valid_to,area_sqkm,version,lang,lang1,lang2,lang3,adm2_ref_n,center_lat,center_lon
0,"POLYGON ((89.45595 23.01135, 89.45661 23.01113...",Khulna,None,None,None,BD4047,Khulna,None,None,None,...,NaT,4457.26501,v03,en,None,None,None,None,22.365739,89.452816


In [7]:
boundary_files = sorted(BOUNDARY.glob("*.shp"))

if not boundary_files:
    raise FileNotFoundError("No boundary shapefile found.")

khulna_boundary = gpd.read_file(boundary_files[0])

print("Boundary CRS:", khulna_boundary.crs)
print("Features:", len(khulna_boundary))

Boundary CRS: EPSG:4326
Features: 1


In [8]:
from rasterio.mask import mask


def clip_raw_raster(
    input_raster,
    output_raster,
    boundary_gdf,
):
    output_raster.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with rasterio.open(input_raster) as src:

        boundary_in_raster_crs = boundary_gdf.to_crs(
            src.crs
        )

        geometries = [
            geometry
            for geometry in boundary_in_raster_crs.geometry
            if geometry is not None
        ]

        clipped_data, clipped_transform = mask(
            src,
            geometries,
            crop=True,
            filled=True,
            nodata=src.nodata,
        )

        clipped_profile = src.profile.copy()

        clipped_profile.update(
            {
                "height": clipped_data.shape[1],
                "width": clipped_data.shape[2],
                "transform": clipped_transform,
                "compress": "lzw",
            }
        )

        with rasterio.open(
            output_raster,
            "w",
            **clipped_profile
        ) as dst:
            dst.write(clipped_data)

In [9]:
test_input = sorted(CHIRPS.glob("*.tif"))[0]

CLIPPED_TEST_FOLDER = (
    PROCESSED_DATA
    / "test_clip"
)

CLIPPED_TEST_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

test_clip_output = (
    CLIPPED_TEST_FOLDER
    / test_input.name
)

clip_raw_raster(
    input_raster=test_input,
    output_raster=test_clip_output,
    boundary_gdf=khulna_boundary,
)

print("Saved:", test_clip_output)

Saved: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\test_clip\2017_01.tif


In [10]:
with rasterio.open(test_input) as src:
    print("RAW RASTER")
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)

with rasterio.open(test_clip_output) as src:
    print("\nCLIPPED RASTER")
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)

RAW RASTER
Width: 12
Height: 28
Bounds: BoundingBox(left=89.20040802594113, bottom=21.65009903320208, right=89.80041077051024, top=23.05010543719667)
Resolution: (0.050000228714092564, 0.050000228714092564)

CLIPPED RASTER
Width: 12
Height: 28
Bounds: BoundingBox(left=89.20040802594113, bottom=21.65009903320208, right=89.80041077051024, top=23.05010543719667)
Resolution: (0.050000228714092564, 0.050000228714092564)


In [11]:
import numpy as np
import rasterio

with rasterio.open(test_input) as src:
    raw = src.read(1, masked=True)

with rasterio.open(test_clip_output) as src:
    clipped = src.read(1, masked=True)

print("Raw valid pixels     :", raw.count())
print("Clipped valid pixels :", clipped.count())
print("Raw masked pixels    :", np.ma.count_masked(raw))
print("Clipped masked pixels:", np.ma.count_masked(clipped))

Raw valid pixels     : 336
Clipped valid pixels : 336
Raw masked pixels    : 0
Clipped masked pixels: 0


In [12]:
datasets = {
    "CHIRPS": CHIRPS,
    "ERA5": ERA5,
    "CCS": CCS,
    "CDR": CDR,
    "IMERG": IMERG,
    "GSMaP_Gauge": GSMAP,
    "GSMaP_MVK": GSMAP_MVK,
    "NDVI": NDVI,
    "LST": LST,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
}

for name, folder in datasets.items():

    tif = sorted(folder.glob("*.tif"))[0]

    with rasterio.open(tif) as src:

        print("="*60)
        print(name)
        print("CRS       :", src.crs)
        print("Width     :", src.width)
        print("Height    :", src.height)
        print("Resolution:", src.res)
        print("Bounds    :", src.bounds)

CHIRPS
CRS       : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Width     : 12
Height    : 28
Resolution: (0.050000228714092564, 0.050000228714092564)
Bounds    : BoundingBox(left=89.20040802594113, bottom=21.65009903320208, right=89.80041077051024, top=23.05010543719667)
ERA5
CRS       : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Width     : 4
Height    : 7
Resolution: (0.2500011435704628, 0.2500011435704628)
Bounds    : BoundingBox(left=89.00040711108477, bottom=21.5000983470598, right=90.00041168536661, top=23.25010635205304)
CCS
CRS       : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PR

In [13]:
REFERENCE_RASTER = sorted(NDVI.glob("*.tif"))[0]

with rasterio.open(REFERENCE_RASTER) as ref:
    print("Reference:", REFERENCE_RASTER.name)
    print("CRS:", ref.crs)
    print("Shape:", ref.height, ref.width)
    print("Resolution:", ref.res)
    print("Bounds:", ref.bounds)

Reference: NDVI_2017_1.tif
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Shape: 151 59
Resolution: (0.008983152841195215, 0.008983152841195215)
Bounds: BoundingBox(left=89.22965717159208, bottom=21.658381500121664, right=89.7596631892226, top=23.014837579142142)


In [14]:
# ==========================================================
# 03_Raster_Preprocessing.ipynb
# Full Raster Processing Code
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import yaml

from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask


# ==========================================================
# 1. Project Root and Configuration
# ==========================================================

PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna"
)

CONFIG_FILE = PROJECT_ROOT / "environment.yml"

with open(CONFIG_FILE, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("Configuration loaded successfully.")
print("Project root:", PROJECT_ROOT)


# ==========================================================
# 2. Define Paths
# ==========================================================

PROCESSED_DATA = PROJECT_ROOT / config["paths"]["processed_data"]

CCS = PROJECT_ROOT / config["paths"]["ccs"]
CDR = PROJECT_ROOT / config["paths"]["cdr"]
CHIRPS = PROJECT_ROOT / config["paths"]["chirps"]
ERA5 = PROJECT_ROOT / config["paths"]["era5"]

GSMAP = PROJECT_ROOT / config["paths"]["gsmap"]
GSMAP_MVK = PROJECT_ROOT / config["paths"]["gsmap_mvk"]
IMERG = PROJECT_ROOT / config["paths"]["imerg"]

LST = PROJECT_ROOT / config["paths"]["lst"]
NDVI = PROJECT_ROOT / config["paths"]["ndvi"]

PDIR = PROJECT_ROOT / config["paths"]["pdir"]
PERSIANN = PROJECT_ROOT / config["paths"]["persiann"]

LAND_VARIABLE = PROJECT_ROOT / config["paths"]["land_variable"]
DISTANCE_SEA = PROJECT_ROOT / config["paths"]["distance_sea"]
BOUNDARY = PROJECT_ROOT / config["paths"]["boundary"]

ALIGNED_OUTPUT = PROCESSED_DATA / "rasters_aligned"
LOG_OUTPUT = PROCESSED_DATA / "logs"

ALIGNED_OUTPUT.mkdir(parents=True, exist_ok=True)
LOG_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Aligned output:", ALIGNED_OUTPUT)
print("Log output:", LOG_OUTPUT)


# ==========================================================
# 3. Load Khulna Boundary
# ==========================================================

boundary_files = sorted(BOUNDARY.glob("*.shp"))

if not boundary_files:
    raise FileNotFoundError(
        f"No boundary shapefile found in: {BOUNDARY}"
    )

boundary_file = boundary_files[0]

khulna_boundary = gpd.read_file(boundary_file)

if khulna_boundary.crs is None:
    raise ValueError(
        "Khulna boundary does not have a CRS."
    )

khulna_boundary = khulna_boundary[
    khulna_boundary.geometry.notna()
].copy()

khulna_boundary = khulna_boundary.dissolve()

print("Boundary file:", boundary_file.name)
print("Boundary CRS:", khulna_boundary.crs)


# ==========================================================
# 4. Select Reference Raster
# ==========================================================
# NDVI has approximately 1 km resolution.

ndvi_files = sorted(
    list(NDVI.glob("*.tif")) +
    list(NDVI.glob("*.tiff"))
)

if not ndvi_files:
    raise FileNotFoundError(
        f"No NDVI TIFF file found in: {NDVI}"
    )

REFERENCE_RASTER = ndvi_files[0]

with rasterio.open(REFERENCE_RASTER) as ref:
    reference_crs = ref.crs
    reference_transform = ref.transform
    reference_width = ref.width
    reference_height = ref.height
    reference_bounds = ref.bounds
    reference_resolution = ref.res

if reference_crs is None:
    raise ValueError(
        "Reference raster does not have a CRS."
    )

print("\nReference raster information")
print("-" * 60)
print("File:", REFERENCE_RASTER.name)
print("CRS:", reference_crs)
print("Width:", reference_width)
print("Height:", reference_height)
print("Resolution:", reference_resolution)
print("Bounds:", reference_bounds)


# ==========================================================
# 5. Dataset Definitions
# ==========================================================

monthly_datasets = {
    "CCS": CCS,
    "CDR": CDR,
    "CHIRPS": CHIRPS,
    "ERA5": ERA5,
    "GSMaP_Gauge": GSMAP,
    "GSMaP_MVK": GSMAP_MVK,
    "IMERG": IMERG,
    "LST": LST,
    "NDVI": NDVI,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
}


# ==========================================================
# 6. Static Raster Definitions
# ==========================================================

static_datasets = {}

distance_meter = DISTANCE_SEA / "Distance_Sea_meter.tif"

if distance_meter.exists():
    static_datasets["Distance_Sea_meter"] = distance_meter
else:
    print(
        "Warning: Distance_Sea_meter.tif was not found."
    )

land_variable_files = sorted(
    list(LAND_VARIABLE.glob("*.tif")) +
    list(LAND_VARIABLE.glob("*.tiff"))
)

for raster_file in land_variable_files:
    static_datasets[
        f"Land_{raster_file.stem}"
    ] = raster_file

print("\nMonthly datasets:", len(monthly_datasets))
print("Static rasters:", len(static_datasets))


# ==========================================================
# 7. Resampling Selection
# ==========================================================

def select_resampling(dataset_name, file_name=""):
    """
    Use nearest-neighbour for categorical rasters.
    Use bilinear interpolation for continuous rasters.
    """

    text = f"{dataset_name} {file_name}".lower()

    categorical_keywords = [
        "landcover",
        "land_cover",
        "lulc",
        "landuse",
        "land_use",
        "class",
        "category",
        "soil_type",
    ]

    if any(
        keyword in text
        for keyword in categorical_keywords
    ):
        return Resampling.nearest

    return Resampling.bilinear


# ==========================================================
# 8. Raster Processing Function
# ==========================================================

def process_raster(
    input_raster,
    output_raster,
    reference_raster,
    boundary_gdf,
    resampling_method=Resampling.bilinear,
    override_source_crs=False,
):
    """
    Reproject and resample an input raster to the reference
    grid, then apply the Khulna boundary mask.

    The original raw raster remains unchanged.
    """

    output_raster.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with rasterio.open(reference_raster) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
        ref_profile = ref.profile.copy()

    with rasterio.open(input_raster) as src:
        source_crs = src.crs

        # PDIR metadata contains an incorrect LOCAL_CS.
        # Its coordinates appear to be longitude/latitude.
        if override_source_crs:
            source_crs = ref_crs

        if source_crs is None:
            raise ValueError(
                f"Missing CRS: {input_raster}"
            )

        destination = np.full(
            (ref_height, ref_width),
            np.nan,
            dtype=np.float32,
        )

        reproject(
            source=rasterio.band(src, 1),
            destination=destination,

            src_transform=src.transform,
            src_crs=source_crs,
            src_nodata=src.nodata,

            dst_transform=ref_transform,
            dst_crs=ref_crs,
            dst_nodata=np.nan,

            resampling=resampling_method,
        )

    boundary_reference_crs = boundary_gdf.to_crs(
        ref_crs
    )

    geometries = [
        geometry
        for geometry in boundary_reference_crs.geometry
        if geometry is not None
        and not geometry.is_empty
    ]

    if not geometries:
        raise ValueError(
            "No valid boundary geometry found."
        )

    inside_boundary = geometry_mask(
        geometries,
        out_shape=(
            ref_height,
            ref_width,
        ),
        transform=ref_transform,
        invert=True,
    )

    destination[~inside_boundary] = np.nan

    output_profile = ref_profile.copy()

    output_profile.update(
        {
            "driver": "GTiff",
            "height": ref_height,
            "width": ref_width,
            "transform": ref_transform,
            "crs": ref_crs,
            "count": 1,
            "dtype": "float32",
            "nodata": np.nan,
            "compress": "lzw",
        }
    )

    with rasterio.open(
        output_raster,
        "w",
        **output_profile
    ) as dst:
        dst.write(destination, 1)


# ==========================================================
# 9. Process All Monthly Rasters
# ==========================================================

monthly_success = []
monthly_failed = []

for dataset_name, input_folder in monthly_datasets.items():

    raster_files = sorted(
        list(input_folder.glob("*.tif")) +
        list(input_folder.glob("*.tiff"))
    )

    output_folder = (
        ALIGNED_OUTPUT
        / "monthly"
        / dataset_name
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        f"\nProcessing {dataset_name}: "
        f"{len(raster_files)} file(s)"
    )

    for index, input_file in enumerate(
        raster_files,
        start=1
    ):

        output_file = (
            output_folder
            / input_file.name
        )

        resampling_method = select_resampling(
            dataset_name,
            input_file.name
        )

        override_crs = dataset_name == "PDIR"

        try:
            process_raster(
                input_raster=input_file,
                output_raster=output_file,
                reference_raster=REFERENCE_RASTER,
                boundary_gdf=khulna_boundary,
                resampling_method=resampling_method,
                override_source_crs=override_crs,
            )

            monthly_success.append(
                {
                    "Dataset": dataset_name,
                    "Input_File": input_file.name,
                    "Output_File": str(output_file),
                    "Resampling": resampling_method.name,
                    "CRS_Override": override_crs,
                    "Status": "Success",
                }
            )

            if (
                index == 1
                or index == len(raster_files)
                or index % 12 == 0
            ):
                print(
                    f"  Completed: "
                    f"{index}/{len(raster_files)}"
                )

        except Exception as error:
            monthly_failed.append(
                {
                    "Dataset": dataset_name,
                    "Input_File": input_file.name,
                    "Error": str(error),
                }
            )

            print(
                f"  Failed: {input_file.name}"
            )


# ==========================================================
# 10. Process Static Rasters
# ==========================================================

static_success = []
static_failed = []

static_output_folder = (
    ALIGNED_OUTPUT
    / "static"
)

static_output_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProcessing static rasters")

for variable_name, input_file in static_datasets.items():

    output_file = (
        static_output_folder
        / f"{variable_name}.tif"
    )

    resampling_method = select_resampling(
        variable_name,
        input_file.name
    )

    try:
        process_raster(
            input_raster=input_file,
            output_raster=output_file,
            reference_raster=REFERENCE_RASTER,
            boundary_gdf=khulna_boundary,
            resampling_method=resampling_method,
            override_source_crs=False,
        )

        static_success.append(
            {
                "Variable": variable_name,
                "Input_File": input_file.name,
                "Output_File": str(output_file),
                "Resampling": resampling_method.name,
                "Status": "Success",
            }
        )

        print("  Processed:", variable_name)

    except Exception as error:
        static_failed.append(
            {
                "Variable": variable_name,
                "Input_File": input_file.name,
                "Error": str(error),
            }
        )

        print("  Failed:", variable_name)


# ==========================================================
# 11. Create Processing Logs
# ==========================================================

monthly_success_df = pd.DataFrame(
    monthly_success
)

monthly_failed_df = pd.DataFrame(
    monthly_failed
)

static_success_df = pd.DataFrame(
    static_success
)

static_failed_df = pd.DataFrame(
    static_failed
)


# ==========================================================
# 12. Grid Information Function
# ==========================================================

def get_grid_information(raster_path):
    """
    Return raster grid and valid-pixel information.
    """

    with rasterio.open(raster_path) as src:
        data = src.read(1, masked=True)

        return {
            "File": raster_path.name,
            "Full_Path": str(raster_path),
            "CRS": str(src.crs),
            "Width": src.width,
            "Height": src.height,
            "Resolution_X": abs(src.res[0]),
            "Resolution_Y": abs(src.res[1]),
            "Transform": tuple(src.transform),
            "Bounds": tuple(src.bounds),
            "Valid_Pixels": int(data.count()),
            "Masked_Pixels": int(
                np.ma.count_masked(data)
            ),
        }


# ==========================================================
# 13. Check All Processed Rasters
# ==========================================================

grid_records = []

processed_raster_files = sorted(
    ALIGNED_OUTPUT.rglob("*.tif")
)

for processed_file in processed_raster_files:
    try:
        grid_records.append(
            get_grid_information(processed_file)
        )
    except Exception as error:
        print(
            "Grid check failed:",
            processed_file.name,
            error
        )

grid_check_df = pd.DataFrame(
    grid_records
)


# ==========================================================
# 14. Alignment Summary
# ==========================================================

if grid_check_df.empty:
    alignment_summary = pd.Series(
        {
            "Total Files": 0,
            "Unique CRS": 0,
            "Unique Width": 0,
            "Unique Height": 0,
            "Unique Resolution X": 0,
            "Unique Resolution Y": 0,
            "Unique Transform": 0,
            "Unique Bounds": 0,
        }
    )
else:
    alignment_summary = pd.Series(
        {
            "Total Files": len(grid_check_df),
            "Unique CRS": grid_check_df[
                "CRS"
            ].nunique(),
            "Unique Width": grid_check_df[
                "Width"
            ].nunique(),
            "Unique Height": grid_check_df[
                "Height"
            ].nunique(),
            "Unique Resolution X": grid_check_df[
                "Resolution_X"
            ].nunique(),
            "Unique Resolution Y": grid_check_df[
                "Resolution_Y"
            ].nunique(),
            "Unique Transform": grid_check_df[
                "Transform"
            ].nunique(),
            "Unique Bounds": grid_check_df[
                "Bounds"
            ].nunique(),
        }
    )


# ==========================================================
# 15. Empty Raster Check
# ==========================================================

if grid_check_df.empty:
    empty_rasters_df = pd.DataFrame()
else:
    empty_rasters_df = grid_check_df[
        grid_check_df["Valid_Pixels"] == 0
    ].copy()


# ==========================================================
# 16. Save Logs
# ==========================================================

monthly_success_df.to_csv(
    LOG_OUTPUT
    / "monthly_raster_processing_log.csv",
    index=False
)

grid_check_df.to_csv(
    LOG_OUTPUT
    / "aligned_raster_grid_check.csv",
    index=False
)

if not monthly_failed_df.empty:
    monthly_failed_df.to_csv(
        LOG_OUTPUT
        / "monthly_raster_failed_log.csv",
        index=False
    )

if not static_success_df.empty:
    static_success_df.to_csv(
        LOG_OUTPUT
        / "static_raster_processing_log.csv",
        index=False
    )

if not static_failed_df.empty:
    static_failed_df.to_csv(
        LOG_OUTPUT
        / "static_raster_failed_log.csv",
        index=False
    )

if not empty_rasters_df.empty:
    empty_rasters_df.to_csv(
        LOG_OUTPUT
        / "empty_rasters.csv",
        index=False
    )


# ==========================================================
# 17. Final Report
# ==========================================================

print("\n" + "=" * 80)
print("RASTER PREPROCESSING COMPLETED")
print("=" * 80)

print("Reference raster      :", REFERENCE_RASTER)
print("Output folder         :", ALIGNED_OUTPUT)
print("Monthly successful    :", len(monthly_success_df))
print("Monthly failed        :", len(monthly_failed_df))
print("Static successful     :", len(static_success_df))
print("Static failed         :", len(static_failed_df))
print("Processed raster files:", len(grid_check_df))
print("Empty rasters         :", len(empty_rasters_df))

print("\nAlignment summary")
print("-" * 80)
print(alignment_summary)

print("=" * 80)

if not monthly_failed_df.empty:
    print("\nMonthly failures:")
    display(monthly_failed_df)

if not static_failed_df.empty:
    print("\nStatic failures:")
    display(static_failed_df)

if not empty_rasters_df.empty:
    print("\nEmpty rasters:")
    display(empty_rasters_df)

Configuration loaded successfully.
Project root: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna
Aligned output: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\rasters_aligned
Log output: E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\processed\logs
Boundary file: Khulna.shp
Boundary CRS: EPSG:4326

Reference raster information
------------------------------------------------------------
File: NDVI_2017_1.tif
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Width: 59
Height: 151
Resolution: (0.008983152841195215, 0.008983152841195215)
Bounds: BoundingBox(left=89.22965717159208, bottom=21.658381500121664, right=89.7596631892226, top=23.014837579142142)

Monthly datasets: 11
Static rasters: 4

Processing CCS: 72 file(s

,Dataset,Input_File,Error
0,CDR,2017_01.tif,"Cannot find coordinate operations from '{ ""$..."
1,CDR,2017_02.tif,"Cannot find coordinate operations from '{ ""$..."
2,CDR,2017_03.tif,"Cannot find coordinate operations from '{ ""$..."
3,CDR,2017_04.tif,"Cannot find coordinate operations from '{ ""$..."
4,CDR,2017_05.tif,"Cannot find coordinate operations from '{ ""$..."
...,...,...,...
68,CDR,2022_09.tif,"Cannot find coordinate operations from '{ ""$..."
69,CDR,2022_10.tif,"Cannot find coordinate operations from '{ ""$..."
70,CDR,2022_11.tif,"Cannot find coordinate operations from '{ ""$..."
71,CDR,2022_12.tif,"Cannot find coordinate operations from '{ ""$..."
